In [137]:
import pandas as pd
import numpy as np
import yaml
import geopandas as gpd
from shapely.geometry import Point
import re

In [138]:
path_to_file = "data/mmc-100.yaml"

with open(path_to_file, "r", encoding="utf-8") as file:
    raw_data = yaml.safe_load(file)
    
df = pd.DataFrame(raw_data)

df_naselja = pd.read_csv("./processed_data/naselja.csv")

In [139]:
# 1. Create a unique list of all searchable terms (naselje, rodilnik, mestnik)
# We filter out Nones/NaNs to avoid errors
search_terms = set(df_naselja[['naselje', 'rodilnik', 'mestnik']].values.flatten())
search_terms = {str(term) for term in search_terms if term and str(term).lower() != 'nan'}

# 2. Build a regex pattern for efficiency 
# \b ensures we match whole words only
pattern = '|'.join([re.escape(word) for word in search_terms])
regex = re.compile(rf'\b({pattern})\b', flags=re.IGNORECASE)

# 3. Define a helper to find matching regions
def get_intersected_regions(text_list):
    full_text = " ".join(text_list)
    found_words = set(regex.findall(full_text))
    
    if not found_words:
        return None
    
    # Map found words back to their regions in df_naselja
    # We check if any of the three columns match the found words
    mask = (
        df_naselja['naselje'].isin(found_words) | 
        df_naselja['rodilnik'].isin(found_words) | 
        df_naselja['mestnik'].isin(found_words)
    )
    return list(df_naselja.loc[mask, 'region_name'].unique())

# 4. Apply and filter
df['intersected_regions'] = df['paragraphs'].apply(get_intersected_regions)

In [140]:
novice_z_naselji_df = df[df['intersected_regions'].astype(bool)]

In [142]:
novice_z_naselji_df.count()

_id                    31
url                    31
topics                 31
authors                30
date                   31
figures                31
keywords               31
lead                   31
mention                31
paragraphs             31
title                  31
gpt_keywords           22
id                     31
n_comments             27
category                1
intersected_regions    31
dtype: int64

In [143]:
# Spreminjaj i v "novice_z_naselji_df.loc[i]" da vidiš na katere besede proži
x = novice_z_naselji_df.loc[2]["paragraphs"]
for m in x:
	found_words = set(regex.findall(m))
	print(found_words)

set()
{'Javornik'}
set()
set()
{'Ljubljani'}
set()
set()
set()
set()
set()
set()
set()
set()
set()
set()
set()
set()
set()
set()
set()
set()
set()
set()
